# Reading a satellite image properly

Four questions, in the order that actually settles the problem.  Q1 gets an RGB picture
on screen, and each question after it checks something that picture cannot show.

Everything is read with `rasterio`, which is GDAL with a friendlier interface — Colab
already has it.  The file is a THEOS-2 scene over Bangkok.

In [ ]:
import sys, urllib.request

BASE = 'https://raw.githubusercontent.com/BWMIN-Hub/SR_practice/main'
for f in ('lib/intake.py', 'lib/readcheck.py'):
    urllib.request.urlretrieve(f'{BASE}/{f}', f.split('/')[-1])
for m in ('intake', 'readcheck'):
    sys.modules.pop(m, None)
import numpy as np
import rasterio
from readcheck import *

PATH = urllib.request.urlretrieve(f'{BASE}/dataset/intake/input.tif', 'input.tif')[0]

## Q1.  Am I reading it correctly?

Open it, read it, and put bands 1, 2, 3 on screen.  Does a picture come out?

In [ ]:
with rasterio.open(PATH) as d:
    img = d.read()                     # (band, row, col)

print(img.shape, img.dtype)
print('values', img.min(), '-', img.max())

An array arrived, with four bands and values from 1 to 4095.  Now draw it.

In [ ]:
check_read(PATH)

Drawn as it comes, it is black.  Rescaled to the range the data actually occupies, it
is a city — so yes, the file reads, and RGB does display.

Why the first attempt failed is **Q2**.  Whether `B1 B2 B3` really is red, green, blue
is **Q4**.

## Q2.  What type are the numbers?

Panel 1 was black because `uint16` was taken at face value.

In [ ]:
with rasterio.open(PATH) as d:
    print('dtype', d.dtypes[0])

print('the box holds 0 -', np.iinfo(img.dtype).max)
print('the data uses 0 -', img.max())

In [ ]:
check_type(PATH)

`uint16` is the size of the box, not the amount in it.  The sensor is 12-bit, so the
values stop at 4095 and 94 % of the box is never used.  Stretch to the range the data
actually occupies and panel 1 becomes panel 2.

That also means **the display range has to be chosen**, every time.  It is not a
property of the file.

## Q3.  How big is it?

Two different numbers answer this.

In [ ]:
with rasterio.open(PATH) as d:
    print('pixels    ', d.width, 'x', d.height, ' bands', d.count)
    print('pixel size', d.res)
    print('bounds    ', tuple(round(v, 1) for v in d.bounds), d.crs)

In [ ]:
check_size(PATH)

How much ground it covers decides whether it is the right scene.  How much memory it
takes decides whether it opens at all — on disk it is 21 MB, in memory 46 MB, because
the file is compressed and the array is not.

## Q4.  How are the bands arranged?

Q1 assumed band 1 is red.  Before trusting that, ask the file.

In [ ]:
with rasterio.open(PATH) as d:
    print('bands      ', d.count, d.indexes)
    print('names      ', d.descriptions)
    print('colorinterp', tuple(c.name for c in d.colorinterp))

No names, and `colorinterp` says `gray` followed by `undefined` three times.  **The
file does not know which band is which.**  That has to come from the product spec — or
from looking.

In [ ]:
check_bands(PATH)

In the fourth band the river goes black and the parks go bright, which is what water
and vegetation do to near infrared.  So band 4 is NIR and bands 1 - 3 are red, green,
blue — which is what Q1 assumed.

Guessing the other way raises no error and still looks like an image.

In [ ]:
check_order(PATH)